# Smart MCQ Solver — Milestone 1
**Email:** 23f3001763@ds.study.iitm.ac.in


In [14]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# ── Load dataset ──────────────────────────────────────────────────────────────
# On Kaggle the file lives here:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

print('Shape:', train.shape)
print('Columns:', list(train.columns))
train.head(3)

Shape: (2000, 8)
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C


---
## Q1 — Frequency distribution of correct answers


In [15]:
# ── Q1 ────────────────────────────────────────────────────────────────────────
ans_counts = train['answer'].value_counts()
print('Answer frequency distribution:')
print(ans_counts)
print()

most_freq_count  = ans_counts.max()
least_freq_count = ans_counts.min()
q1_answer = most_freq_count + least_freq_count

print(f'Most frequent option : {ans_counts.idxmax()} ({most_freq_count})')
print(f'Least frequent option: {ans_counts.idxmin()} ({least_freq_count})')
print(f'\n>>> Q1 Answer (sum) = {q1_answer}')

Answer frequency distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent option : B (490)
Least frequent option: E (324)

>>> Q1 Answer (sum) = 814


---
## Q2 — Vocabulary size of cleaned prompt column


In [16]:
# ── Q2 ────────────────────────────────────────────────────────────────────────
def clean_text(text):
    """Lowercase + remove string.punctuation + split on whitespace."""
    text = str(text).lower()
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    return text

train['prompt_clean'] = train['prompt'].apply(clean_text)

unique_words = set()
for line in train['prompt_clean']:
    unique_words.update(line.split())

q2_answer = len(unique_words)
print(f'>>> Q2 Answer (vocabulary size of cleaned prompts) = {q2_answer}')

>>> Q2 Answer (vocabulary size of cleaned prompts) = 859


---
## Q3 — Stop-word filtered word count for Row ID 1


In [17]:
# ── Q3 ────────────────────────────────────────────────────────────────────────
# Row ID 1 means the row where the 'id' column == 1
row1 = train[train['id'] == 1].iloc[0]

words_row1 = row1['prompt_clean'].split()
filtered_words = [w for w in words_row1 if w not in ENGLISH_STOP_WORDS]

print('Original cleaned words :', words_row1)
print()
print('After removing stop words:', filtered_words)
print()
q3_answer = len(filtered_words)
print(f'>>> Q3 Answer (words left after stop-word removal for Row ID 1) = {q3_answer}')

Original cleaned words : ['pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heideggers', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options']

After removing stop words: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']

>>> Q3 Answer (words left after stop-word removal for Row ID 1) = 13


---
## Q4 — TF-IDF vocabulary size


In [18]:
# ── Q4 ────────────────────────────────────────────────────────────────────────
# Build a list: one document per row = prompt + A + B + C + D + E concatenated
combined_docs = (
    train['prompt'].fillna('') + ' ' +
    train['A'].fillna('') + ' ' +
    train['B'].fillna('') + ' ' +
    train['C'].fillna('') + ' ' +
    train['D'].fillna('') + ' ' +
    train['E'].fillna('')
).tolist()

tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(combined_docs)

q4_answer = len(tfidf.vocabulary_)
print(f'>>> Q4 Answer (TF-IDF vocabulary size) = {q4_answer}')

>>> Q4 Answer (TF-IDF vocabulary size) = 2762


---
## Q5 — Cosine similarity between prompt and option A for Row ID 1


In [19]:
# ── Q5 ────────────────────────────────────────────────────────────────────────
prompt_vec  = tfidf.transform([row1['prompt']])
option_a_vec = tfidf.transform([row1['A']])

sim_score = cosine_similarity(prompt_vec, option_a_vec)[0][0]
q5_answer = round(sim_score, 4)

print(f'Raw similarity score : {sim_score}')
print(f'>>> Q5 Answer (cosine similarity, Row ID 1, prompt vs option A) = {q5_answer}')

Raw similarity score : 0.27202429519891635
>>> Q5 Answer (cosine similarity, Row ID 1, prompt vs option A) = 0.272


---
## Q6 — Percentage of rows where TF-IDF picks the correct answer


In [20]:
# ── Q6 ────────────────────────────────────────────────────────────────────────
OPTIONS = ['A', 'B', 'C', 'D', 'E']

# Vectorize all prompts and all 5 option columns at once (efficient batch approach)
prompt_vecs = tfidf.transform(train['prompt'].fillna(''))
option_vecs  = {opt: tfidf.transform(train[opt].fillna('')) for opt in OPTIONS}

# Build similarity matrix: shape (N, 5)
sim_matrix = np.column_stack([
    cosine_similarity(prompt_vecs, option_vecs[opt]).diagonal()
    for opt in OPTIONS
])

# Pick the option index with the highest similarity
best_option_idx  = np.argmax(sim_matrix, axis=1)
best_option_pred = [OPTIONS[i] for i in best_option_idx]

correct_mask = [pred == gt for pred, gt in zip(best_option_pred, train['answer'])]
q6_answer = round(sum(correct_mask) / len(correct_mask) * 100, 4)

print(f'Correct predictions: {sum(correct_mask)} / {len(correct_mask)}')
print(f'>>> Q6 Answer (% where highest TF-IDF similarity == correct answer) = {q6_answer}%')

Correct predictions: 271 / 2000
>>> Q6 Answer (% where highest TF-IDF similarity == correct answer) = 13.55%


---
## Q7 — MAP@3: Ground truth = C, prediction = [C, A, B]


In [21]:
# ── MAP@3 Helper ──────────────────────────────────────────────────────────────
def apk(actual, predicted, k=3):
    """Average Precision at K for a single question."""
    score = 0.0
    for rank, pred in enumerate(predicted[:k], start=1):
        if pred == actual:
            score = 1.0 / rank
            break   # only one correct answer
    return score

def mapk(actuals, predictions, k=3):
    """Mean Average Precision at K across all questions."""
    return np.mean([apk(a, p, k) for a, p in zip(actuals, predictions)])

# ── Q7 ────────────────────────────────────────────────────────────────────────
q7_answer = apk('C', ['C', 'A', 'B'])
print(f'Ground truth = C   |   Prediction = [C, A, B]')
print(f'Correct answer found at rank 1  →  1/1 = {q7_answer}')
print(f'>>> Q7 Answer (MAP@3) = {q7_answer}')

Ground truth = C   |   Prediction = [C, A, B]
Correct answer found at rank 1  →  1/1 = 1.0
>>> Q7 Answer (MAP@3) = 1.0


---
## Q8 — MAP@3: Ground truth = B, prediction = [D, B, E]


In [22]:
# ── Q8 ────────────────────────────────────────────────────────────────────────
q8_answer = apk('B', ['D', 'B', 'E'])
print(f'Ground truth = B   |   Prediction = [D, B, E]')
print(f'Correct answer found at rank 2  →  1/2 = {q8_answer}')
print(f'>>> Q8 Answer (MAP@3) = {q8_answer}')

Ground truth = B   |   Prediction = [D, B, E]
Correct answer found at rank 2  →  1/2 = 0.5
>>> Q8 Answer (MAP@3) = 0.5


---
## Q9 — Majority Class Baseline MAP@3


In [23]:
# ── Q9 ────────────────────────────────────────────────────────────────────────
top3_answers = ans_counts.index[:3].tolist()   # From Q1
print(f'Top-3 most frequent answers: {top3_answers}')

majority_preds = [top3_answers] * len(train)
q9_answer = round(mapk(train['answer'].tolist(), majority_preds), 4)

print(f'>>> Q9 Answer (Majority Class Baseline MAP@3) = {q9_answer}')

Top-3 most frequent answers: ['B', 'C', 'A']
>>> Q9 Answer (Majority Class Baseline MAP@3) = 0.4212


---
## Q10 — TF-IDF Pipeline MAP@3


In [24]:
# ── Q10 ───────────────────────────────────────────────────────────────────────
# sim_matrix already computed in Q6: shape (N, 5) where columns = [A, B, C, D, E]

# For each row, argsort similarities descending to get top-3
sorted_indices = np.argsort(-sim_matrix, axis=1)          # shape (N, 5)
top3_indices   = sorted_indices[:, :3]                    # shape (N, 3)
tfidf_preds    = [[OPTIONS[i] for i in row] for row in top3_indices]

q10_answer = round(mapk(train['answer'].tolist(), tfidf_preds), 4)
print(f'>>> Q10 Answer (TF-IDF Pipeline MAP@3) = {q10_answer}')

>>> Q10 Answer (TF-IDF Pipeline MAP@3) = 0.2915


---
## Summary of All Answers

In [25]:
# ── SUMMARY ───────────────────────────────────────────────────────────────────
print('=' * 55)
print('         MILESTONE 1 — ANSWER SUMMARY')
print('=' * 55)
print(f' Q1  | Sum (most + least frequent answers)  : {q1_answer}')
print(f' Q2  | Vocabulary size (cleaned prompts)    : {q2_answer}')
print(f' Q3  | Words left after stop-word filter    : {q3_answer}')
print(f' Q4  | TF-IDF vocabulary size               : {q4_answer}')
print(f' Q5  | Cosine sim (Row 1, prompt vs opt A)  : {q5_answer}')
print(f' Q6  | % rows: TF-IDF picks correct answer  : {q6_answer}%')
print(f' Q7  | MAP@3 (GT=C, pred=[C,A,B])           : {q7_answer}')
print(f' Q8  | MAP@3 (GT=B, pred=[D,B,E])           : {q8_answer}')
print(f' Q9  | Majority Class Baseline MAP@3        : {q9_answer}')
print(f' Q10 | TF-IDF Pipeline MAP@3                : {q10_answer}')
print('=' * 55)

         MILESTONE 1 — ANSWER SUMMARY
 Q1  | Sum (most + least frequent answers)  : 814
 Q2  | Vocabulary size (cleaned prompts)    : 859
 Q3  | Words left after stop-word filter    : 13
 Q4  | TF-IDF vocabulary size               : 2762
 Q5  | Cosine sim (Row 1, prompt vs opt A)  : 0.272
 Q6  | % rows: TF-IDF picks correct answer  : 13.55%
 Q7  | MAP@3 (GT=C, pred=[C,A,B])           : 1.0
 Q8  | MAP@3 (GT=B, pred=[D,B,E])           : 0.5
 Q9  | Majority Class Baseline MAP@3        : 0.4212
 Q10 | TF-IDF Pipeline MAP@3                : 0.2915
